<a href="https://colab.research.google.com/github/fboldt/aulasml/blob/master/aula09a_sele%C3%A7%C3%A3o_de_caracter%C3%ADsticas.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split

X, y = fetch_openml('arrhythmia', version=1, return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(X_train.shape)
print(X_test.shape)


(361, 279)
(91, 279)


In [2]:
from sklearn.impute import SimpleImputer

imputer = SimpleImputer(strategy='median')
X_train = imputer.fit_transform(X_train)
X_test = imputer.transform(X_test)

In [3]:
from sklearn.neighbors import KNeighborsClassifier

model = KNeighborsClassifier()

def evaluate_model(model, data):
    X_train, X_test, y_train, y_test = data
    model.fit(X_train, y_train)
    score = model.score(X_test, y_test)
    return score

data = (X_train, X_test, y_train, y_test)
score = evaluate_model(model, data)
print(score)

0.5604395604395604


In [4]:
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

scaler = StandardScaler()
model_std = make_pipeline(scaler, model)
score_std = evaluate_model(model_std, data)
print(score_std)

0.5494505494505495


# PCA

In [5]:
from sklearn.decomposition import PCA

pca = PCA(n_components=0.99)
model_pca = make_pipeline(pca, model)
score_pca = evaluate_model(model_pca, data)
print(score_pca)

0.5604395604395604


In [6]:
n_feats = model_pca.steps[0][1].n_components_
print(n_feats)

63


In [7]:
model_std_pca = make_pipeline(scaler, pca, model)
score_std_pca = evaluate_model(model_std_pca, data)
print(score_std_pca)

0.5384615384615384


In [8]:
model_pca_std = make_pipeline(pca, scaler, model)
score_pca_std = evaluate_model(model_pca_std, data)
print(score_pca_std)

0.4835164835164835


# Feature Selection

## Ranking

### Filter

In [9]:
import numpy as np

def variance_score(X, y=None):
  return np.var(X, axis=0).mean()

variance_score(X_train)

np.float64(158.87840499935655)

In [10]:
def feature_ranking(X, y=None, score_function=variance_score):
  scores = []
  for f in range(X.shape[1]):
    score = score_function(X[:, f:f+1], y)
    scores.append(score)
  sorted_idxs = np.argsort(scores)[::-1]
  return sorted_idxs

feature_ranking(X_train)

array([ 10,  13,   9,   5,   2,   7, 248,  12, 258,   6, 247,  11,  56,
        87, 238,  51,   8,  41, 237, 268, 257,  89,  40,  64,  65,  77,
       149,  39,  99,  29, 227,  76,  17,  53, 278,  16,  44, 178,  92,
        63,  28, 188, 101, 228, 137,  52,   3, 125,   0, 267, 218, 187,
       113,   4,  75,  68, 111, 148, 177, 100, 217,  88, 277, 167,  14,
       168,  80, 207, 198, 112, 208,  90, 104, 136, 197,  27, 140, 152,
        54, 123,  20, 147, 135,  32,  15, 124, 116,  42, 102, 128, 242,
       232, 251, 252, 241,  66, 222, 261,  78, 230, 211,  43, 271, 171,
       181, 231, 138, 220, 114, 190,  79, 246, 240, 161, 201, 262, 192,
       126, 182,  30, 256, 236, 272, 127, 221, 202, 212, 180, 103, 172,
       226, 150, 266,  91,  55, 162, 239, 276, 176, 166, 115, 191, 223,
        18, 186, 229, 196, 219, 250, 216, 249, 206, 210, 200, 183, 260,
       233, 259, 159, 269,  31, 225, 199, 179, 175, 185, 170, 245, 189,
       169, 213, 270, 215, 235,   1, 209, 255, 193, 205, 195, 16

In [11]:
from sklearn.base import BaseEstimator, TransformerMixin

class FeatureSelector(BaseEstimator, TransformerMixin):
  def __init__(self, n_feats=None,
               score_function=variance_score,
               ranking_function=feature_ranking):
    self.n_feats = n_feats
    self.score_function = score_function
    self.ranking_function = ranking_function

  def fit(self, X, y=None):
    if self.n_feats is None:
      self.n_feats = X.shape[1]
    ranking = self.ranking_function(X, y, self.score_function)
    self.feats = ranking[:self.n_feats]
    return self

  def transform(self, X):
    return X[:, self.feats]

feat_rank = FeatureSelector(n_feats=n_feats)
model_feat_rank = make_pipeline(feat_rank, model)
score_feat_rank = evaluate_model(model_feat_rank, data)
print(score_feat_rank)

0.5494505494505495


In [12]:
class HybridSelector(BaseEstimator, TransformerMixin):
  def __init__(self, model,
               score_function=variance_score,
               ranking_function=feature_ranking):
    self.model = model
    self.score_function = score_function
    self.ranking_function = ranking_function

  def fit(self, X, y=None):
    X_tr, X_va, y_tr, y_va = train_test_split(X, y, test_size=0.2, random_state=42)
    ranking = self.ranking_function(X_tr, y_tr, self.score_function)
    best_score = -np.inf
    best_n = 0
    for n in range(1, ranking.shape[0] + 1):
      feats = ranking[:n]
      model = self.model.fit(X_tr[:, feats], y_tr)
      score = model.score(X_va[:, feats], y_va)
      if score > best_score:
        best_score = score
        best_n = n
    self.feats = ranking[:best_n]
    return self

  def transform(self, X):
    return X[:, self.feats]

hybrid_selector = HybridSelector(model)
model_hybrid = make_pipeline(hybrid_selector, model)
score_hybrid = evaluate_model(model_hybrid, data)
print(score_hybrid)

0.5714285714285714


In [13]:
len(hybrid_selector.feats)

12

In [14]:
hybrid_selector.feats

array([ 10,  13,   5,   2,   9, 248,   7,  12, 258,   6, 247,  11])

## distance score

In [15]:
import numpy as np

TMP = np.array([
    [2, 1, 3],
    [5, 6, 4],
    [4, 0, 8],
    [5, 9, 3],
])

def pairwise_distances(X1, X2):
  return np.linalg.norm(X1[:, None, :] - X2[None, :, :], axis=-1)

pairwise_distances(TMP, TMP)

array([[ 0.        ,  5.91607978,  5.47722558,  8.54400375],
       [ 5.91607978,  0.        ,  7.28010989,  3.16227766],
       [ 5.47722558,  7.28010989,  0.        , 10.34408043],
       [ 8.54400375,  3.16227766, 10.34408043,  0.        ]])

In [16]:
for f in range(TMP.shape[1]):
  print(f"Feature {f}")
  dists = pairwise_distances(TMP[:, f:f+1], TMP[:, f:f+1])
  print(f"Mean distance: {dists.mean()}")
  print()

Feature 0
Mean distance: 1.25

Feature 1
Mean distance: 4.0

Feature 2
Mean distance: 2.0



In [17]:
def intra_class_mean_distance(X, y):
  mean_dist = 0
  for c in np.unique(y):
    X_c = X[y == c]
    mean_dist += pairwise_distances(X_c, X_c).mean()
  return mean_dist / len(np.unique(y))

y_tmp = [0,1,0,1]
intra_class_mean_distance(TMP, y_tmp)

np.float64(2.15987580880501)

In [18]:
def inter_class_mean_distance(X, y):
  mean_dist = 0
  for c in np.unique(y):
    X_in = X[y == c]
    X_out = X[y != c]
    mean_dist += pairwise_distances(X_in, X_out).mean()
  return mean_dist / len(np.unique(y))

inter_class_mean_distance(TMP, y_tmp)

np.float64(8.021068462621567)

In [19]:
def distance_score(X, y):
  return inter_class_mean_distance(X, y) / (intra_class_mean_distance(X, y)+1e-8)

distance_score(TMP, y_tmp)

np.float64(3.7136711253424592)

In [20]:
feat_rank_dist = HybridSelector(model, score_function=distance_score)
model_feat_rank_dist = make_pipeline(feat_rank_dist, model)
score_feat_rank_dist = evaluate_model(model_feat_rank_dist, data)
print(score_feat_rank_dist)

0.5934065934065934


In [21]:
len(feat_rank_dist.feats)

55

In [22]:
feat_rank_dist.feats

array([ 48, 150,  23,  37,  47,  59, 153,  45, 155,  71,  73,  85,  97,
        84, 130, 273,  43, 113,   5,  16, 184,   4, 243, 111, 148, 237,
         8,  41, 229, 227,  39,  18, 114,  52,  56, 277,  27,  14, 275,
       265,  90, 170,  20, 247, 236,  51,   7,  66, 112,  95, 163,  28,
       257,  63,   6])

## wrapper score

In [23]:
def knn_acc_score(X, y):
  model = KNeighborsClassifier()
  X_tr, X_va, y_tr, y_va = train_test_split(X, y, test_size=0.2, random_state=42)
  model.fit(X_tr, y_tr)
  return model.score(X_va, y_va)

feat_rank_knn = HybridSelector(model, score_function=knn_acc_score)
model_rank_knn = make_pipeline(feat_rank_knn, model)
score_rank_knn = evaluate_model(model_rank_knn, data)
print(score_rank_knn)

0.5824175824175825


In [24]:
len(feat_rank_knn.feats)

28

In [25]:
feat_rank_knn.feats

array([227, 240, 242, 223, 113,  90, 256, 112, 241,  75, 210, 206, 251,
       246, 124, 230, 248, 250,  20,  27, 170, 123, 152, 148,  14, 266,
       128,  16])

# SFS - Sequential Forward Selection

In [26]:
def sequential_forward_selection(X, y, score_function):
  best_combination = []
  remaining_features = list(range(X.shape[1]))
  for _ in range(X.shape[1]):
    best_score = -np.inf
    best_feature = None
    for f in remaining_features:
      new_combination = best_combination + [f]
      score = score_function(X[:, new_combination], y)
      if score > best_score:
        best_score = score
        best_feature = f
    best_combination.append(best_feature)
    remaining_features.remove(best_feature)
  return np.array(best_combination)

sequential_forward_selection(TMP, y_tmp, distance_score)

array([1, 0, 2])

In [27]:
feat_sfs_var = HybridSelector(model,
                              score_function=variance_score,
                              ranking_function=sequential_forward_selection)
model_feat_sfs_var = make_pipeline(feat_sfs_var, model)
score_feat_sfs_var = evaluate_model(model_feat_sfs_var, data)
print(score_feat_sfs_var)

0.5714285714285714


In [28]:
len(feat_sfs_var.feats)

12

In [29]:
feat_sfs_var.feats

array([ 10,  13,   5,   2,   9, 248,   7,  12, 258,   6, 247,  11])

In [33]:
feat_sfs_dist = HybridSelector(model,
                               score_function=distance_score,
                               ranking_function=sequential_forward_selection)
model_feat_sfs_dist = make_pipeline(feat_sfs_dist, model)
# score_feat_sfs_dist = evaluate_model(model_feat_sfs_dist, data)
# print(score_feat_sfs_dist)
# len(feat_sfs_dist)

In [32]:
feat_sfs_knn = HybridSelector(model,
                            score_function=knn_acc_score,
                            ranking_function=sequential_forward_selection)
model_feat_sfs_knn = make_pipeline(feat_sfs_knn, model)
score_feat_sfs_knn = evaluate_model(model_feat_sfs_knn, data)
print(score_feat_sfs_knn)
print(len(feat_sfs_knn.feats))

0.6373626373626373
133


# Cross Validation

In [37]:
import warnings
warnings.filterwarnings('ignore')

In [38]:
from sklearn.model_selection import cross_validate

scores = cross_validate(model, X_train, y_train, cv=5, return_train_score=True)
def print_scores(scores):
  for s in scores:
    print(f"{s}: {scores[s].mean()}")

print_scores(scores)

fit_time: 0.0029349327087402344
score_time: 0.003493976593017578
test_score: 0.5956240487062405
train_score: 0.6502811418685122


In [40]:
scores = cross_validate(model_feat_rank_dist, X_train, y_train, cv=5, return_train_score=True)
print_scores(scores)

fit_time: 2.7595338344573976
score_time: 0.00313720703125
test_score: 0.6288432267884322
train_score: 0.6994569396386006


In [ ]:
scores = cross_validate(model_feat_sfs_knn, X_train, y_train, cv=5, return_train_score=True)
print_scores(scores)

In [ ]:
scores = cross_validate(model_feat_sfs_dist, X_train, y_train, cv=5, return_train_score=True)
print_scores(scores)

# Outros Modelos

In [42]:
from sklearn.ensemble import RandomForestClassifier

model_rf = RandomForestClassifier()
scores = cross_validate(model_rf, X_train, y_train, cv=5, return_train_score=True)
print_scores(scores)

fit_time: 0.4203339099884033
score_time: 0.012367439270019532
test_score: 0.7174657534246576
train_score: 1.0
